# LoRA pretraining: modality-local representation report

This notebook compares **student** representations from LoRA pretraining checkpoints. It deliberately does not measure image-text alignment. Text and image are evaluated in separate forward passes.

Metrics are produced by `evaluate_jepa_modality.py`:

- direct shared/private clean-to-masked cosine at exactly masked positions;
- JEPA predictor-to-clean-target cosine when a checkpoint has JEPA;
- count and per-attribute balanced accuracy from clean-trained, masked-input linear probes;
- exact original token-ID recovery from shared/private token features.

`best` always means the maximum over layers 2, 3, and 4. It is not an average across layers.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('/home/zd25e122/clevr_discrete_diffusion')
TEXT = ROOT / 'outputs/jepa_modality_eval_lora_family_text/results.json'
IMAGE = ROOT / 'outputs/jepa_modality_eval_lora_family_image/results.json'

def load(path):
    if not path.exists():
        raise FileNotFoundError(f'{path} is not available yet. Run the matching evaluator first.')
    return json.loads(path.read_text())

text_report = load(TEXT)
image_report = load(IMAGE)


In [ ]:
def semantic_table(report, mask='t0.8', branch='shared'):
    rows = []
    for model, item in report['models'].items():
        for layer, value in item['semantic_clean_train_masked_test'][mask].items():
            probe = value[branch]
            row = {'model': model, 'layer': int(layer),
                   'count': probe['object_count']['balanced_accuracy'],
                   'attribute_mean': probe['attribute_group_mean']}
            row.update(probe['attribute_balanced_accuracy'])
            rows.append(row)
    return pd.DataFrame(rows).sort_values(['model', 'layer'])

text_shared_80 = semantic_table(text_report)
image_shared_80 = semantic_table(image_report)
display(text_shared_80)
display(image_shared_80)


In [ ]:
def best_by_model(frame, score='attribute_mean'):
    return (frame.sort_values(score, ascending=False)
                 .groupby('model', as_index=False).first()
                 .sort_values(score, ascending=False))

print('Text: best shared layer per checkpoint at 80% masking')
display(best_by_model(text_shared_80)[['model', 'layer', 'count', 'attribute_mean']])
print('Image: best shared layer per checkpoint at 80% masking')
display(best_by_model(image_shared_80)[['model', 'layer', 'count', 'attribute_mean']])

# Individual-attribute plot; change `text_shared_80` to image_shared_80 if desired.
attributes = ['gray','red','blue','green','brown','purple','cyan','yellow',
              'cube','sphere','cylinder','metal','rubber','small','large',
              'left','right','front','behind']
plot = best_by_model(text_shared_80).set_index('model')[attributes].T
ax = plot.plot(kind='bar', figsize=(15, 5), width=0.85)
ax.set_ylabel('balanced accuracy; clean-trained probe, 80% masked input')
ax.set_xlabel('attribute')
plt.tight_layout()


In [ ]:
def masking_curve(report, model, layer, branch='shared'):
    rows = []
    for ratio, layers in report['models'][model]['semantic_clean_train_masked_test'].items():
        x = layers[str(layer)][branch]
        rows.append({'mask_ratio': float(ratio[1:]),
                     'count': x['object_count']['balanced_accuracy'],
                     'attribute_mean': x['attribute_group_mean']})
    return pd.DataFrame(rows).sort_values('mask_ratio')

# Example: compare every model's count robustness at layer 4.
fig, ax = plt.subplots(figsize=(7, 4))
for name in text_report['models']:
    curve = masking_curve(text_report, name, layer=4)
    ax.plot(curve.mask_ratio, curve['count'], marker='o', label=name)
ax.set(xlabel='mask ratio', ylabel='count balanced accuracy', title='Text, shared layer 4')
ax.legend(bbox_to_anchor=(1.04, 1), loc='upper left')
plt.tight_layout()

# For exact-token detail and direct JEPA scores, inspect the same layer/ratio:
model, layer, ratio = 'plain_lora_no_stage', '4', 't0.8'
item = text_report['models'][model]
display(pd.Series(item['direct_masked_position_stability'][ratio][layer]))
display(pd.DataFrame(item['exact_token_clean_train_masked_test'][ratio][layer]).T)
